# MIDA507 -- Session 05
## Qualite des Donnees : Framework DAMA et Nettoyage Professionnel

| | |
|---|---|
| **Programme** | Master MIDA -- Universite de Douala |
| **Session** | S05 -- Qualite des donnees et nettoyage Python |
| **Duree estimee** | 90 minutes |
| **Prerequis** | Notebooks S01 a S04 |

### Ce que vous allez apprendre
1. Pourquoi la qualite des donnees est une responsabilite centrale de l'IDA
2. Les 6 dimensions DAMA : le cadre international d'evaluation de la qualite
3. Profiler automatiquement un jeu de donnees selon ces 6 dimensions
4. Appliquer 5 regles de nettoyage Python expliquees pas a pas
5. Produire un rapport de qualite structure

### Livrable : `MIDA507_S05_rapport_qualite.json` + jeu nettoye


In [ ]:
!pip install pandas seaborn matplotlib openpyxl --quiet
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import json, os, random
from datetime import datetime, timedelta
plt.rcParams['figure.figsize']=(12,5)
sns.set_theme(style='whitegrid')
print('Outils prets.')


In [ ]:
random.seed(42)
NB=5000
TYPES=["These et memoire","Manuel universitaire","Revue scientifique",
       "Rapport de recherche","Ouvrage de reference","Document administratif"]
FIL=["Droit","Sciences eco.","Lettres","Histoire","Geographie","Medecine","Agronomie","Informatique"]
NIV=["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
REG=["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
LNG=["Francais","Anglais","Bilingue","Arabe","Autres"]
d0=datetime(2018,1,1)
emprunts=pd.DataFrame({
    "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(NB)],
    "type_doc":random.choices(TYPES,weights=[0.28,0.30,0.15,0.10,0.10,0.07],k=NB),
    "date":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(NB)],
    "duree":random.choices([3,7,7,14,14,14,21,30],k=NB),
    "usager":[f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(NB)],
    "filiere":random.choices(FIL,k=NB),
    "niveau":random.choices(NIV,k=NB),
    "region":random.choices(REG,k=NB),
    "langue":random.choices(LNG,weights=[0.62,0.22,0.10,0.04,0.02],k=NB),
})
emprunts["annee"]=pd.to_datetime(emprunts["date"]).dt.year
# On introduit des problèmes realistes comme dans un vrai fichier d'institution
import numpy as np
rng=np.random.default_rng(99)
idx_vides=rng.choice(NB,size=int(NB*0.06),replace=False)
emprunts.loc[idx_vides,"langue"]=None            # 6% de langues manquantes
idx_inv=rng.choice(NB,size=int(NB*0.04),replace=False)
emprunts.loc[idx_inv,"filiere"]="non renseigne"  # 4% de filieres invalides
idx_aber=rng.choice(NB,size=int(NB*0.02),replace=False)
emprunts.loc[idx_aber,"duree"]=999               # 2% de durees aberrantes
# 3 doublons intentionnels
emprunts=pd.concat([emprunts, emprunts.iloc[[10,50,100]]], ignore_index=True)
print(f"Jeu BU-UNG avec problemes de qualite : {len(emprunts):,} lignes, {len(emprunts.columns)} colonnes.")
print("Ce jeu contient volontairement des problemes de qualite -- c'est normal pour cet exercice.")


---
## NOTION 1 -- La Qualite des Donnees : Definition et Enjeux

### Definition en 3 lignes

La **qualite des donnees** mesure dans quelle mesure un jeu de donnees est apte a remplir l'usage pour lequel il a ete collecte. Des donnees de mauvaise qualite publiees en open data nuisent a la reputation de l'institution et induisent des erreurs chez les reutilisateurs.

**Analogie IDA :** C'est l'equivalent du **controle de catalogage** en bibliotheque : verifier qu'une notice n'est pas incomplete (zone obligatoire vide), incorrecte (date impossible), ou dupliquee (double saisie). L'IDA fait ce controle depuis toujours -- il faut maintenant le formaliser pour les donnees numeriques.

### Pourquoi c'est critique AVANT la publication ?

Un jeu publie avec des erreurs est **tres difficile a corriger** une fois en ligne : les reutilisateurs ont deja telecharge la version fautive. Comme en archivistique, la prevention est plus efficace que la correction a posteriori.

### Le Framework DAMA : 6 dimensions de qualite

| Dimension | Question cle | Equivalent IDA |
|---|---|---|
| **Completude** | Y a-t-il des cases vides ? | Zone obligatoire non renseignee en catalogage |
| **Unicite** | Y a-t-il des doublons ? | Double saisie dans un fichier d'autorite |
| **Exactitude** | Les valeurs sont-elles plausibles ? | Date de publication impossiblements en avant |
| **Validite** | Les valeurs respectent-elles le domaine autorise ? | Genre document non liste dans le referentiel |
| **Coherence** | Les champs sont-ils coherents entre eux ? | Date retour avant date emprunt |
| **Actualite** | Les donnees sont-elles recentes ? | Fonds non mis a jour depuis 5 ans |


In [ ]:
# NOTION 1 EN PRATIQUE -- Premier regard sur les problemes du jeu
print("PREMIER DIAGNOSTIC -- Jeu BU-UNG (avec problemes de qualite)")
print("=" * 60)
print()
print(f"  Nombre de lignes     : {len(emprunts):,}")
print(f"  Nombre de colonnes   : {len(emprunts.columns)}")
print()
print("  APERCU DES 5 PREMIERES LIGNES :")
print(emprunts.head(5).to_string(max_cols=6))
print()
print("  RESUME RAPIDE DES TYPES DE DONNEES :")
for col in emprunts.columns:
    nb_vides = emprunts[col].isna().sum()
    nb_uniques = emprunts[col].nunique()
    indicateur = ""
    if nb_vides > 0:
        indicateur = f"  <-- {nb_vides} case(s) vide(s) !"
    print(f"    {col:<20} : {emprunts[col].dtype}  |  {nb_uniques} valeurs distinctes{indicateur}")
print()
print("  PREMIERE QUESTION IDA : Ces donnees sont-elles publiables en l'etat ?")
print("  REPONSE : NON -- il faut auditer et nettoyer avant de publier.")


---
## NOTION 2 -- Les 6 Dimensions DAMA Expliquees et Mesurees

### En 3 lignes

On va maintenant mesurer chaque dimension sur notre jeu. Pour chaque dimension, Python calcule un score de 0 a 100. L'IDA interprete ce score et decide des actions correctives.

**Important :** Python fait les calculs, mais **c'est l'IDA qui decide ce qui est "valide"** : quels genres documentaires sont autorises, quelle duree d'emprunt est raisonnable, quelles regions existent au Cameroun. Ce savoir metier ne peut pas venir d'un algorithme.


In [ ]:
# NOTION 2 EN PRATIQUE -- Profil DAMA des 6 dimensions
from datetime import date

rapport_dama = {}

# ────────────────────────────────────────────────────────────
# DIMENSION 1 : COMPLETUDE
# Question : combien de cases sont vides dans notre tableau ?
# ────────────────────────────────────────────────────────────
# .isnull() = True si la case est vide, False sinon
# .mean() = proportion de True dans chaque colonne
# 1 - proportion_vide = proportion remplie = completude

completude_par_colonne = 1 - emprunts.isnull().mean()
completude_globale = completude_par_colonne.mean()
rapport_dama["completude"] = {
    "score": round(float(completude_globale)*100, 1),
    "colonnes_problematiques": [c for c,v in completude_par_colonne.items() if v < 0.95],
    "conforme": completude_globale >= 0.95
}

print("DIMENSION 1 -- COMPLETUDE")
print("  Question : Combien de cases sont vides dans notre tableau ?")
print(f"  Score global : {completude_globale*100:.1f}%")
print("  Detail par colonne :")
for col, val in completude_par_colonne.items():
    barre = chr(9608)*int(val*20) + chr(9617)*(20-int(val*20))
    statut = "OK" if val >= 0.95 else "!"
    print(f"    [{statut}] {col:<20} : {val*100:.1f}%  {barre}")
print()

# ────────────────────────────────────────────────────────────
# DIMENSION 2 : UNICITE
# Question : y a-t-il des enregistrements dupliques ?
# ────────────────────────────────────────────────────────────
# .duplicated() = True pour chaque ligne qui est une copie exacte d'une precedente
nb_doublons = emprunts.duplicated().sum()
rapport_dama["unicite"] = {
    "score": round((1 - nb_doublons/len(emprunts))*100, 1),
    "nb_doublons": int(nb_doublons),
    "conforme": nb_doublons == 0
}
print("DIMENSION 2 -- UNICITE")
print("  Question : Y a-t-il des enregistrements identiques (doublons) ?")
if nb_doublons == 0:
    print("  OK  Aucun doublon detecte -- score 100%")
else:
    print(f"  !   {nb_doublons} doublon(s) detecte(s) -- score {rapport_dama['unicite']['score']}%")
    print(f"      Exemple de doublon (lignes identiques) :")
    doublons = emprunts[emprunts.duplicated(keep=False)].head(2)
    print(doublons[["cote","date","usager"]].to_string())
print()


In [ ]:
# ────────────────────────────────────────────────────────────
# DIMENSION 3 : EXACTITUDE
# Question : Les valeurs numeriques sont-elles plausibles ?
# ────────────────────────────────────────────────────────────
# On definit les bornes valides pour la duree d'emprunt
# Ici : entre 1 et 90 jours (limite raisonnable pour une bibliotheque)
# .between(1, 90) = True si la valeur est dans la plage

masque_duree_valide = emprunts["duree"].between(1, 90)
nb_aberrants = (~masque_duree_valide).sum()
score_exactitude = masque_duree_valide.mean() * 100

rapport_dama["exactitude"] = {
    "score": round(float(score_exactitude), 1),
    "nb_aberrants": int(nb_aberrants),
    "conforme": score_exactitude >= 97
}
print("DIMENSION 3 -- EXACTITUDE")
print("  Question : Les durees d'emprunt sont-elles plausibles (1 a 90 jours) ?")
print(f"  Score : {score_exactitude:.1f}%")
if nb_aberrants > 0:
    aberrants = emprunts[~masque_duree_valide][["cote","date","duree"]].head(5)
    print(f"  !  {nb_aberrants} valeur(s) hors limite (duree > 90 jours ou < 1 jour) :")
    print(aberrants.to_string(index=False))
    print()
    print("  NOTE IDA : La valeur '999' signifie souvent 'donnee manquante mal codee'")
    print("  par l'agent collecteur. C'est un probleme de formation, pas de saisie.")
print()

# ────────────────────────────────────────────────────────────
# DIMENSION 4 : VALIDITE
# Question : Les genres documentaires respectent-ils le referentiel ?
# ────────────────────────────────────────────────────────────
# L'IDA definit les valeurs autorisees : c'est le dictionnaire de donnees
GENRES_VALIDES = {"These et memoire","Manuel universitaire","Revue scientifique",
                  "Rapport de recherche","Ouvrage de reference","Document administratif"}
FILIERES_VALIDES = {"Droit","Sciences eco.","Lettres","Histoire","Geographie",
                    "Medecine","Agronomie","Informatique"}

masque_genres_ok = emprunts["type_doc"].isin(GENRES_VALIDES)
masque_filieres_ok = emprunts["filiere"].isin(FILIERES_VALIDES)
score_validite = ((masque_genres_ok & masque_filieres_ok).mean()) * 100

rapport_dama["validite"] = {
    "score": round(float(score_validite), 1),
    "genres_invalides": emprunts[~masque_genres_ok]["type_doc"].value_counts().to_dict(),
    "filieres_invalides": emprunts[~masque_filieres_ok]["filiere"].value_counts().head(5).to_dict(),
    "conforme": score_validite >= 95
}
print("DIMENSION 4 -- VALIDITE")
print("  Question : Les valeurs appartiennent-elles aux referentiels autorises ?")
print(f"  Score : {score_validite:.1f}%")
nb_fil_inv = (~masque_filieres_ok).sum()
if nb_fil_inv > 0:
    print(f"  !  {nb_fil_inv} filiere(s) invalide(s) detectee(s) :")
    for val, nb in rapport_dama["validite"]["filieres_invalides"].items():
        print(f"     '{val}' : {nb} occurences")
    print("     --> Recommandation IDA : remplacer 'non renseigne' par une categorie valide")
    print("         ou creer la categorie 'Non communique' dans le referentiel.")
print()


In [ ]:
# ────────────────────────────────────────────────────────────
# DIMENSIONS 5 ET 6 : COHERENCE ET ACTUALITE
# ────────────────────────────────────────────────────────────
# Coherence : les filières invalides croisees avec le niveau
# Actualite : date du dernier enregistrement vs aujourd'hui

# COHERENCE : Une filiere 'non renseigne' avec un niveau 'Master 2' = incoherent
masque_coherent = ~((emprunts["filiere"]=="non renseigne") & (emprunts["niveau"]=="Master 2"))
score_coherence = masque_coherent.mean() * 100

# ACTUALITE : Combien de jours depuis le dernier enregistrement ?
try:
    derniere_date = pd.to_datetime(emprunts["date"]).max()
    jours_retard = (pd.Timestamp(date.today()) - derniere_date).days
    score_actualite = max(0, 100 - jours_retard / 7.3)  # perte de 1 pt par semaine
except:
    jours_retard = 0; score_actualite = 100.0

rapport_dama["coherence"] = {
    "score": round(float(score_coherence), 1),
    "conforme": score_coherence >= 99
}
rapport_dama["actualite"] = {
    "score": round(float(score_actualite), 1),
    "jours_depuis_derniere_donnee": jours_retard,
    "conforme": jours_retard <= 365,
    "note": "Le jeu couvre 2018-2023 -- actualisation annuelle recommandee"
}

print("DIMENSION 5 -- COHERENCE")
print(f"  Score : {score_coherence:.1f}%")
print()
print("DIMENSION 6 -- ACTUALITE")
print(f"  Dernier enregistrement : {pd.to_datetime(emprunts['date']).max().strftime('%Y-%m-%d')}")
print(f"  Jours depuis la derniere donnee : {jours_retard}")
print(f"  Score : {score_actualite:.1f}%")
print()

# Bilan global DAMA
print("BILAN DAMA -- Toutes les 6 dimensions")
print("=" * 60)
for dim, val in rapport_dama.items():
    score = val["score"]
    conforme = val["conforme"]
    barre = chr(9608)*int(score//5) + chr(9617)*(20-int(score//5))
    statut = "OK" if conforme else "A CORRIGER"
    print(f"  [{statut:10}] {dim.capitalize():<15} : {score:.1f}/100  {barre}")

score_global = round(sum(v["score"] for v in rapport_dama.values()) / len(rapport_dama), 1)
print()
print(f"  SCORE GLOBAL DAMA : {score_global}/100")
niv = "Excellent (publiable)" if score_global>=95 else ("Bon" if score_global>=85 else ("Passable" if score_global>=70 else "Insuffisant"))
print(f"  Niveau : {niv}")
print()
print("  CONCLUSION : Ce jeu necessite des corrections avant publication.")
print("  Nous allons maintenant appliquer les 5 regles de nettoyage.")


---
## NOTION 3 -- Nettoyage Professionnel : 5 Regles Appliquees

### En 3 lignes

Le **nettoyage des donnees** est l'application systematique de regles de correction aux problemes identifies par le profil DAMA. Chaque regle est documentee dans le journal de lineage (session S02) pour garantir la tracabilite.

**Analogie IDA :** C'est le **travail de correction de catalogage** -- reprendre des notices incorrectes ou incompletes et les mettre aux normes. En version numerique, Python applique la meme correction sur 5 000 lignes en 1 seconde au lieu de manuellement.

**Principe cardinal :** On ne modifie jamais les donnees originales. On cree toujours une copie (`emprunts_nettoye`) et on documente chaque transformation.


In [ ]:
# NOTION 3 EN PRATIQUE -- Nettoyage en 5 regles documentees
log_nettoyage = []

# On fait une copie -- on ne touche JAMAIS aux donnees originales
emprunts_nettoye = emprunts.copy()
print("NETTOYAGE PROFESSIONNEL -- 5 REGLES")
print("=" * 60)
print(f"  Avant nettoyage : {len(emprunts_nettoye):,} lignes")
print()

# REGLE 1 -- Supprimer les doublons exacts
n_avant = len(emprunts_nettoye)
emprunts_nettoye = emprunts_nettoye.drop_duplicates()
n_supprimes = n_avant - len(emprunts_nettoye)
log_nettoyage.append({
    "regle": "R1 -- Suppression des doublons",
    "dimension_dama": "Unicite",
    "action": f"Suppression de {n_supprimes} ligne(s) identiques",
    "raison_IDA": "Un meme emprunt ne peut etre enregistre qu'une seule fois",
    "nb_lignes_avant": n_avant,
    "nb_lignes_apres": len(emprunts_nettoye)
})
print(f"  R1 [Unicite]     : {n_supprimes} doublon(s) supprime(s)")

# REGLE 2 -- Corriger les durees aberrantes
masque_aberrant = emprunts_nettoye["duree"] > 90
nb_corriges_r2 = masque_aberrant.sum()
# On remplace les valeurs > 90 par la mediane (valeur centrale representive)
mediane_duree = emprunts_nettoye[~masque_aberrant]["duree"].median()
emprunts_nettoye.loc[masque_aberrant, "duree"] = mediane_duree
log_nettoyage.append({
    "regle": "R2 -- Correction durees aberrantes",
    "dimension_dama": "Exactitude",
    "action": f"Remplacement de {nb_corriges_r2} valeur(s) >90j par la mediane ({mediane_duree:.0f}j)",
    "raison_IDA": "Une duree d'emprunt >90 jours est hors politique de pret de la BU-UNG"
})
print(f"  R2 [Exactitude]  : {nb_corriges_r2} duree(s) aberrante(s) remplacee(s) par {mediane_duree:.0f} jours")

# REGLE 3 -- Remplacer les filieres invalides
FILIERES_VALIDES = {"Droit","Sciences eco.","Lettres","Histoire","Geographie",
                    "Medecine","Agronomie","Informatique"}
masque_inv = ~emprunts_nettoye["filiere"].isin(FILIERES_VALIDES)
nb_corriges_r3 = masque_inv.sum()
emprunts_nettoye.loc[masque_inv, "filiere"] = "Non communique"
log_nettoyage.append({
    "regle": "R3 -- Standardisation filieres invalides",
    "dimension_dama": "Validite",
    "action": f"Remplacement de {nb_corriges_r3} valeur(s) invalide(s) par 'Non communique'",
    "raison_IDA": "Le referentiel IDA ne reconnait pas 'non renseigne' comme filiere valide"
})
print(f"  R3 [Validite]    : {nb_corriges_r3} filiere(s) invalide(s) standardisee(s)")

# REGLE 4 -- Combler les langues manquantes
nb_vides_r4 = emprunts_nettoye["langue"].isna().sum()
emprunts_nettoye["langue"] = emprunts_nettoye["langue"].fillna("Non renseignee")
log_nettoyage.append({
    "regle": "R4 -- Completude langues manquantes",
    "dimension_dama": "Completude",
    "action": f"Remplacement de {nb_vides_r4} case(s) vide(s) par 'Non renseignee'",
    "raison_IDA": "DCAT exige que tous les champs soient renseignes -- une case vide n'est pas informative"
})
print(f"  R4 [Completude]  : {nb_vides_r4} langue(s) manquante(s) remplacee(s)")

# REGLE 5 -- Normaliser les noms de colonnes (conformite DCAT)
noms_avant = list(emprunts_nettoye.columns)
emprunts_nettoye.columns = [c.lower().replace(" ","_") for c in emprunts_nettoye.columns]
log_nettoyage.append({
    "regle": "R5 -- Normalisation noms de colonnes",
    "dimension_dama": "Coherence",
    "action": "Minuscules + underscores (conformite DCAT et CKAN)",
    "raison_IDA": "Les portails CKAN et les APIs JSON n'acceptent pas les espaces dans les noms de colonnes"
})
print(f"  R5 [Coherence]   : Noms de colonnes normalises (minuscules + underscores)")

print()
print(f"  APRES NETTOYAGE : {len(emprunts_nettoye):,} lignes")
print()

# Sauvegarde du jeu nettoye et du journal
emprunts_nettoye.to_csv("MIDA507_S05_jeu_nettoye.csv", index=False, encoding="utf-8-sig")
with open("MIDA507_S05_rapport_qualite.json","w",encoding="utf-8") as f:
    json.dump({"bilan_dama":rapport_dama,"regles_nettoyage":log_nettoyage,
               "nb_lignes_avant":len(emprunts),"nb_lignes_apres":len(emprunts_nettoye),
               "date":datetime.now().strftime("%Y-%m-%d")}, f, ensure_ascii=False, indent=2)
print("OK Jeu nettoye sauvegarde       : MIDA507_S05_jeu_nettoye.csv")
print("OK Rapport qualite sauvegarde   : MIDA507_S05_rapport_qualite.json")


In [ ]:
# Visualisation avant/apres nettoyage
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Graphique 1 : Scores DAMA avant nettoyage
dims = list(rapport_dama.keys())
scores = [rapport_dama[d]["score"] for d in dims]
barres = axes[0].bar([d.capitalize() for d in dims], scores,
                     color=["#2E7D32" if s>=95 else "#E65100" if s>=80 else "#CC0000" for s in scores],
                     edgecolor="white", width=0.6)
axes[0].axhline(y=95, color="#BF360C", linestyle="--", alpha=0.6, label="Seuil qualite (95%)")
axes[0].set_title("Scores DAMA AVANT nettoyage", fontweight="bold", color="#3E1500")
axes[0].set_ylabel("Score (%)")
axes[0].set_ylim(0, 110)
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend()
for barre, score in zip(barres, scores):
    axes[0].text(barre.get_x()+barre.get_width()/2, barre.get_height()+0.5,
                 f"{score:.0f}%", ha="center", fontsize=8)

# Graphique 2 : Comparaison nb lignes avant/apres
categories = ["Avant nettoyage","Apres nettoyage"]
valeurs = [len(emprunts), len(emprunts_nettoye)]
axes[1].bar(categories, valeurs, color=["#546E7A","#2E7D32"], edgecolor="white", width=0.4)
axes[1].set_title("Impact du nettoyage sur le nombre de lignes", fontweight="bold", color="#3E1500")
axes[1].set_ylabel("Nombre d'enregistrements")
for i, v in enumerate(valeurs):
    axes[1].text(i, v+5, f"{v:,}", ha="center", fontweight="bold")
axes[1].set_ylim(0, max(valeurs)*1.1)

plt.suptitle("Session S05 -- Qualite DAMA : Avant vs Apres Nettoyage",
             fontsize=13, fontweight="bold", color="#3E1500", y=1.01)
plt.tight_layout()
plt.show()
print("INTERPRETATION :")
print(f"  {len(emprunts):,} lignes avant --> {len(emprunts_nettoye):,} lignes apres")
print(f"  {len(emprunts)-len(emprunts_nettoye)} ligne(s) supprimee(s) (doublons)")
print(f"  {sum(r.get('nb_corriges',0) for r in [{'nb_corriges':nb_corriges_r2},{'nb_corriges':nb_corriges_r3},{'nb_corriges':nb_vides_r4}])} valeur(s) corrigee(s) sans suppression")


---
## EXERCICE -- Estimez la qualite DAMA de votre institution


In [ ]:
# EXERCICE -- Profil DAMA de mon institution
print("ESTIMATION DAMA -- Mon institution")
print("=" * 50)
MON_JEU = "[A COMPLETER : Nom du jeu de donnees]"
# Estimez chaque dimension de 0 a 100 (approximation)
mes_scores_dama = {
    "Completude":  None,   # % de cases remplies (estimation)
    "Unicite":     None,   # % sans doublons (estimation)
    "Exactitude":  None,   # % de valeurs plausibles (estimation)
    "Validite":    None,   # % de valeurs dans le referentiel (estimation)
    "Coherence":   None,   # % de champs coherents entre eux (estimation)
    "Actualite":   None,   # % (0=donnees > 5 ans, 100=donnees recentes)
}
MON_PROBLEME = "[A COMPLETER : principal probleme qualite identifie]"
MA_SOLUTION  = "[A COMPLETER : regle de nettoyage prioritaire proposee]"

valides = {k:v for k,v in mes_scores_dama.items() if isinstance(v,(int,float)) and 0<=v<=100}
if valides:
    print(f"  Jeu : {MON_JEU}")
    print()
    for dim, score in mes_scores_dama.items():
        if score is not None:
            barre = chr(9608)*int(score//5) + chr(9617)*(20-int(score//5))
            statut = "OK" if score>=95 else ("~" if score>=80 else "!")
            print(f"  [{statut}] {dim:<15} : {score:.0f}/100  {barre}")
    score_global = sum(valides.values())/len(valides)
    print()
    print(f"  Score global estime : {score_global:.0f}/100")
    print(f"  Probleme principal  : {MON_PROBLEME}")
    print(f"  Solution proposee   : {MA_SOLUTION}")
else:
    print("[A COMPLETER] Entrez des estimations de 0 a 100 pour chaque dimension.")
    print()
    print("Guide pour estimer :")
    print("  Completude  : regardez si des colonnes ont des cases vides souvent")
    print("  Unicite     : y a-t-il des enregistrements saisis deux fois ?")
    print("  Exactitude  : y a-t-il des valeurs impossibles (dates futures, ages negatifs) ?")
    print("  Validite    : les categories respectent-elles un referentiel defini ?")
    print("  Coherence   : les champs sont-ils compatibles entre eux ?")
    print("  Actualite   : quand a ete fait le dernier enregistrement ?")


---
## Bilan de la Session 05

| Competence | Acquise |
|---|---|
| Definir la qualite des donnees et son importance pour l'IDA | OK |
| Connaitre les 6 dimensions DAMA avec analogies IDA | OK |
| Profiler automatiquement un jeu de donnees (6 dimensions) | OK |
| Appliquer 5 regles de nettoyage documentees | OK |
| Produire un rapport de qualite structure | OK |
| Estimer la qualite de mon propre jeu | A completer |

### Prochaine session
**S06 -- Publication CKAN :** mettre notre jeu nettoye en ligne sur un portail open data.

*Notebook MIDA507 S05 -- Master MIDA -- Universite de Douala*
